# ROM-FlameBench: a simple walkthrough

This notebook follows one complete compression and forecasting experiment:

1. Prepare and load the data.
2. Compress the fields into a small latent state.
3. Learn how that state evolves under the inlet forcing.
4. Evaluate recursive forecasts on unseen trajectories.

Run the cells in order, from the repository root, using the project's Python environment.
POD fitting and full test rollouts use the real dataset and can take time: this is a
walkthrough of an experiment, not a tiny synthetic demo.

**Confirmed:** snapshot zero is the initial steady state.  
**TODO:** add physical cell volumes. Until then, we evaluate fields only; integrated
heat release and its gain/phase metrics are disabled.

## 1. Data preparation

Raw archives contain `data` with shape **(cell, field, time)**. Preparation writes
float32 `.npy` images with shape **(time, field, 206, 104)** after mesh mapping. These files can be read from
disk in small batches from `Data/Images/Training` and `Data/Images/Test`. Existing prepared files are reused.

The supplied `phi` files contain the dimensionless forcing: $U(t)=U_{base}\phi(t)$.
The metadata defines filenames, field order, units and the sampling interval.

In [ ]:
import json
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader

from DataProcessing.prepare import prepare
from DataProcessing.metadata import load_metadata
from DataProcessing.Dataset import CompressorDataset, ForecasterDataset
from utils import seed_everything, write_json
from Experiments.paths import new_run_name, run_directory

compressor_name = "pod"  # Choose "pod", "cae", or "vit_ae".
device = "cuda" if torch.cuda.is_available() else "cpu"
seed = 42
seed_everything(seed)
torch.set_num_threads(4)

metadata_path = Path("Data/metadata.json")
raw_directory = Path("Data/Raw")
run_settings = {
    "output": "Experiments/Results",
    "seed": seed,
    "compressor": {"name": compressor_name},
    "model": {"name": "arx"},
}
run_settings["run_name"] = new_run_name(run_settings)
output = run_directory(run_settings)
print("Run directory:", output)

Nx = 9  # Past snapshots besides the current one: 10 in total.
Ni = 4  # Past forcing values besides the next one: 5 in total.
rank = 16
batch_size = 64
validation_fraction = 0.3
blocks = 20  # Each training sweep is cut into this many blocks in time.

# .env can enable online W&B logging for the FireMark team.
logging_config = {"logging": {"wandb": {"mode": "offline", "entity": "FireMark"}}}
output.mkdir(parents=True, exist_ok=True)

In [ ]:
prepare(metadata_path, raw_directory)
metadata = load_metadata(metadata_path)

print("Fields:", metadata["fields"])
print("Sampling interval:", metadata["dt"], "seconds")
for case in metadata["cases"]:
    print(case["split"], case["name"])

Each training sweep is cut in time into **`blocks` equal blocks**. **30%** of the blocks,
evenly spaced, are validation; the others are training. Both partitions therefore cover the
whole sweep, low and high frequencies. Consecutive blocks form one segment, and samples never
cross a segment edge, so no frame is used by both partitions. Test simulations are kept separate.

There are two kinds of dataset. A `CompressorDataset` sample is one frame. A `ForecasterDataset`
sample is a window of `max(Nx, Ni) + 1` rows ending at time $t$: row $s$ holds the state $x(s)$ and
the forcing deviation $\phi(s+1)-1$, so the last row pairs $x(t)$ with the known next forcing.
**Nx and Ni are independent**: the states older than the last `Nx + 1` rows and the forcing
older than the last `Ni + 1` rows are set to zero. The target is $x(t+1)$. The `Dataset` defines
samples; a `DataLoader` groups them into batches.

In [ ]:
split = dict(validation_fraction=validation_fraction, blocks=blocks)
frames = CompressorDataset(metadata, "train", **split)
windows = ForecasterDataset(metadata, "train", Nx=Nx, Ni=Ni, **split)
testing = ForecasterDataset(metadata, "test", Nx=Nx, Ni=Ni)

sample = windows[0]
print("Training frames:", len(frames))
print("Training windows:", len(windows))
print("Test windows:", len(testing))
print("States:", tuple(sample["states"].shape))  # (max(Nx, Ni) + 1, fields, height, width)
print("Forcing:", sample["forcing"].tolist())   # phi(s+1) - 1 for each row
print("Target:", tuple(sample["target"].shape))  # (1, fields, height, width)

## 2. Compressor

The fields have different units and scales. First, we estimate a mean and standard
deviation for each field **using training data only and excluding empty pixels**. The datasets
then return scaled frames. Scaling keeps one field from
dominating simply because its numerical values are larger.

POD maps each scaled snapshot to `rank` coefficients using the legacy randomized
SVD (oversampling 20, 7 iterations, seed 42). It first gathers image pixels in original
cell order and subtracts the temporal mean per coordinate. The full training matrix is
loaded into RAM and centered before SVD, as in the legacy code. No volume weights are used.

Choose `cae` for spatial convolutions or `vit_ae` for the same convolutions followed by attention between feature-map pixels. Both compress each snapshot to the same latent interface. The forecaster still receives latent snapshots together with forcing. CLI configs log compressor training to TensorBoard and W&B.

In [ ]:
from DataProcessing.scaling import FeatureScaler
from Baselines.OrderReduction.Linear.POD import POD
from Baselines.OrderReduction.DL.CAE import CAE
from Baselines.OrderReduction.DL.ViTAE import ViTAE

scaler = FeatureScaler(frames.mask)
scaler.fit(DataLoader(frames, batch_size=batch_size))
training = CompressorDataset(metadata, "train", scaler=scaler, **split)
validation = CompressorDataset(metadata, "validation", scaler=scaler, **split)

if compressor_name == "pod":
    compressor = POD(rank=rank, batch_size=batch_size)
elif compressor_name == "cae":
    compressor = CAE(rank=rank, channels=[16, 32, 64], epochs=20, batch_size=16, device=device)
elif compressor_name == "vit_ae":
    compressor = ViTAE(rank=rank, channels=[16, 32, 64], hidden=64, heads=4, layers=2,
                       epochs=20, batch_size=8, device=device)
else:
    raise ValueError(compressor_name)
compressor.fit(training, validation=validation)
print("Latent coefficients per snapshot:", compressor.rank)

`encode` converts a scaled field to coefficients; `decode` reconstructs the scaled
field. Inverse scaling returns the original physical units. This reconstruction check
measures compression alone, before any forecasting.

In [ ]:
scaled = np.stack([validation[0].numpy(), validation[1].numpy()])
latent = compressor.encode(scaled)
reconstructed = scaler.inverse(compressor.decode(latent))

print("Field batch:", scaled.shape)
print("Latent batch:", latent.shape)
print("Reconstructed batch:", reconstructed.shape)
print("Scaled reconstruction MSE:", np.mean((compressor.decode(latent) - scaled)[..., training.mask] ** 2) if training.mask is not None else np.mean((compressor.decode(latent) - scaled) ** 2))

## 3. Forecast

Given a compressor, a `ForecasterDataset` encodes its frames once and keeps the small latent
trajectories in memory. This avoids re-encoding large fields at every training step.

**ARX** is a linear regression from a history of POD coefficients and the prescribed
forcing history (including the next prescribed input) to the next coefficients:

$$\widehat z_{t+1} = f(z_{t-N_x},\ldots,z_t,\phi_{t+1-N_i},\ldots,\phi_{t+1}).$$

`alpha` controls ridge regularization. ARX uses a direct fit, so it has no epochs.
The same latent datasets can also train a GRU, LSTM or Transformer. In those models,
compressed snapshots **and forcing enter the recurrent/attention block together**,
one row per time step. Zero padding handles different `Nx` and `Ni`. The compressor processes fields only.

In [ ]:
from Baselines.Forecast.Classical.ARX import ARX

windows = dict(Nx=Nx, Ni=Ni, scaler=scaler, **split)
train_latent = ForecasterDataset(metadata, "train", compressor=compressor, **windows)
validation_latent = ForecasterDataset(metadata, "validation", compressor=compressor, **windows)

train_loader = DataLoader(train_latent, batch_size=batch_size, shuffle=False)
validation_loader = DataLoader(validation_latent, batch_size=batch_size, shuffle=False)

model = ARX(Nx=Nx, Ni=Ni, alpha=1e-4)
model.fit(train_loader, validation_loader)

Validation checks a **recursive rollout**, not just one-step prediction. It starts
after enough context for both histories inside each validation segment and then feeds predictions
back into the model. The test set does not participate in this step.

This latent MSE is suitable for choosing forecasters with the same compressor. To
compare different compressors or ranks, use a shared field-space validation metric.

In [ ]:
from Experiments.run import validation_rollout, dump

validation_error = validation_rollout(model, compressor, ForecasterDataset(metadata, "validation", **windows))
if not np.isfinite(validation_error):
    raise RuntimeError("Validation rollout diverged. Review the model before testing.")

print("Recursive validation field MSE (scaled, valid pixels):", validation_error)

# Save the fitted objects so they can be reused without fitting again.
dump(output / "preprocessing.pkl", (scaler, compressor))
dump(output / "model.pkl", (compressor, model))

## 4. Evaluation

Each test rollout starts from **snapshot zero**, repeated to fill the model's history.
No later ground-truth field is supplied to the model. Forcing before time zero is
padded with **1**; from time zero onward, we use the supplied forcing. In step cases,
`phi[0]` is already perturbed. No forcing after the next prediction time is used.

The evaluator repeats this sequence until the end of each test simulation:

1. Predict the next latent state from the latent history and $[\phi_{t+1-N_i},\ldots,\phi_{t+1}]$.
2. Decode it and return to physical units.
3. Compare with the reference field and update the history with the prediction.

For each field, NRMSE is RMSE divided by the ground-truth standard deviation, pooling
all evaluated cells and times in that case. The reported mean is the average of the
11 field scores. Snapshot zero is excluded because it was given to the model.

**TODO — cell volumes:** `heat_release=False` explicitly skips integrated $Q(t)$ and
its gain/phase metrics. Local `mix:Q` is still included in the field scores.

In [ ]:
from Experiments.evaluation import evaluate

results = evaluate(
    model=model,
    dataset=testing,
    compressor=compressor,
    scaler=scaler,
    directory=output,
    initialization="steady",
    heat_release=False,  # TODO: supply physical cell volumes before enabling Q(t).
    save_predictions=False,
)

In [ ]:
for name, metrics in results.items():
    print(f"\n{name}")
    print("Mean NRMSE:", metrics["mean_nrmse"])
    for field, error in metrics["field_nrmse"].items():
        print(f"  {field}: {error}")
    print("Seconds per forecast step:", metrics["seconds_per_step"])

Finally, save the settings and send the scalar results to **TensorBoard and W&B**.
W&B defaults to offline here; the project's `.env` can set `WANDB_MODE=online` and
provide the API key for `FireMark`. Credentials are not saved with these settings.

The detailed metrics are already in `metrics.json` inside the seed folder. TensorBoard can read
the run with `tensorboard --logdir Experiments/Results`.
Running the setup cell creates a new `pod_arx_<timestamp>/seed_<seed>` directory.
Re-running later cells uses that same seed directory. CLI multi-seed runs share one
timestamp across seeds with `--seeds 0 1 2`.

In [ ]:
from Experiments.logging import ExperimentLogger

settings = {
    "Nx": Nx,
    "Ni": Ni,
    "rank": rank,
    "batch_size": batch_size,
    "validation_fraction": validation_fraction,
    "compressor": compressor_name,
    "model": "arx",
    "alpha": 1e-4,
    "seed": seed,
    "run_name": run_settings["run_name"],
    "heat_release": False,
    **logging_config,
}
write_json(output / "notebook_settings.json", settings)

logger = ExperimentLogger(output, settings)
try:
    logger.log({"validation/rollout_field_mse": validation_error}, step=0)
    for name, metrics in results.items():
        values = {f"test/{name}/mean_nrmse": metrics["mean_nrmse"]}
        for field, error in metrics["field_nrmse"].items():
            values[f"test/{name}/nrmse/{field}"] = error
        logger.log(values, step=1)
finally:
    logger.close()